In [1]:
import numpy as np
import pandas as pd




In [2]:
import torch 



x = torch.tensor([1,2,3])
x.unsqueeze(1)

tensor([[1],
        [2],
        [3]])

In [3]:
import os
from autograder.dataset import CPEN455_2025_W1_Dataset

dataset_path = os.path.join("autograder/cpen455_released_datasets/merged.csv")
dataset = CPEN455_2025_W1_Dataset(dataset_path)

# Get dataset size
print(len(dataset))
print(dataset[0])

120
(1, 're : meeting follow up', "john ,\n\nthanks for the update on yesterday ' s meeting .\nplease circulate the minutes to the trading team\nand make sure the action items are assigned\nbefore friday noon .\n\nregards ,\nsusan", 0)


### Ideas

* Try out changing the loss function to softmax CE so it's more natural for classification
* Try out adding some dropout layers to the feed foraward networsk to prevent overfitting
* Implement bagging + voting to classify output
* Need to research on prefix tuning 

### Observation
* Softmax CE definitely improved the results, and due to the lower number of training samples, probably gonna go with SGD with low batch size
* Bagging may help a bit, need to put dropout layer

In [4]:
batch_size = 4
max_seq_len = 256
dataset_path = "autograder/cpen455_released_datasets/train_val_subset.csv"
test_dataset_path = "autograder/cpen455_released_datasets/test_subset.csv"
prob_output_folder = "bayes_inverse_probs"
num_iterations = 100
learning_rate = 1e-5


# Ensemble parameters
num_models = 3
bagging = True

In [5]:

import os
import pdb
import wandb
from dotenv import load_dotenv
from einops import rearrange
from tqdm import tqdm
import argparse

import torch
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torch.optim import Optimizer

from autograder.dataset import CPEN455_2025_W1_Dataset, ENRON_LABEL_INDEX_MAP, prepare_subset
from model import LlamaModel
from utils.weight_utils import load_model_weights
from model.config import Config
from model.tokenizer import Tokenizer
from utils.download import _resolve_snapshot_path
from utils.device import set_device

from utils.prompt_template import get_prompt
from utils.logger import avg_logger, avg_acc_logger

/home/kenrickmh/UBC/CPEN455/CPEN455-Project-2025W1/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/kenrickmh/UBC/CPEN455/CPEN455-Project-2025W1/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata 

Initialize environment and wandb

In [6]:
load_dotenv()


True

In [7]:
device = set_device()

Using device: cuda


In [8]:
def get_seq_log_prob(prompts, tokenizer, model, device):
    encoded_batch = tokenizer.encode(
        prompts, return_tensors="pt", return_attention_mask=True
    )
    input_ids = encoded_batch["input_ids"].to(device)
    attention_mask = encoded_batch["attention_mask"].to(device)

    log_prob, _ = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    
    shifted_log_prob = log_prob[:, :-1, :]
    shifted_input_ids = input_ids[:, 1:]
    shifted_attention_mask = attention_mask[:, 1:]

    gathered_log_prob = shifted_log_prob.gather(-1, shifted_input_ids.unsqueeze(-1)).squeeze(-1)
    gathered_log_prob = gathered_log_prob * shifted_attention_mask
    
    return gathered_log_prob.sum(dim=-1)


METHOD_SET = ["zero_shot", "naive_prompting", "full_finetune"]

def is_required_training(method: str) -> bool:
    assert method in METHOD_SET, f"Method {method} not recognized. Choose from {METHOD_SET}."
    return method in METHOD_SET[2:]


def cross_entropy_test(max_seq_length, model, tokenizer, batch, optimizer=None, is_training=True):
    if is_training:
        model.train()
    else:
        model.eval()

    _, subjects, messages, label_indexs = batch
    
    if -1 in label_indexs:
        loss_val = None
    else:

        with torch.no_grad():
            spam_prompts = [get_prompt(subject=subj, message=msg, label=ENRON_LABEL_INDEX_MAP.inv[0], max_seq_length=max_seq_len) for subj, msg in zip(subjects, messages)]
            ham_prompts = [get_prompt(subject=subj, message=msg, label=ENRON_LABEL_INDEX_MAP.inv[1], max_seq_length=max_seq_len) for subj, msg in zip(subjects, messages)]

        # shape: 1xd for both spam and ham
        spam_seq_log_prob = get_seq_log_prob(spam_prompts, tokenizer, model, device=device).cpu()
        ham_seq_log_prob = get_seq_log_prob(ham_prompts, tokenizer, model, device=device).cpu()
    
        # Now, get the posterior probabilities for predicting spam and ham
        softmax_logits = torch.stack((spam_seq_log_prob, ham_seq_log_prob), dim=1)

        # pdb.set_trace()
        ce_loss = torch.nn.CrossEntropyLoss()
        loss_val = ce_loss(softmax_logits, label_indexs)
        
        if is_training:
            assert optimizer is not None, "Optimizer must be provided during training."
            optimizer.zero_grad()
            loss_val.backward()
            optimizer.step()
            
        # Get prediction labels
        labels_pred = torch.argmax(softmax_logits, dim=-1)
        is_correct = labels_pred == label_indexs
        

    # is_correct, (probs, labels_pred) = bayes_inverse_llm_classifier(args, model, batch, tokenizer, device=device)

    return loss_val, is_correct, (softmax_logits.detach().cpu(), labels_pred.detach().cpu())


def bayes_inverse_llm_classifier(model, batch, tokenizer, device, max_seq_len):
    '''
        
    '''

    _, subjects, messages, labels = batch

    prompts_ham = [get_prompt(subject=subj, message=msg, label=ENRON_LABEL_INDEX_MAP.inv[0], max_seq_length=max_seq_len, user_prompt="") for subj, msg in zip(subjects, messages)]
    prompts_spam = [get_prompt(subject=subj, message=msg, label=ENRON_LABEL_INDEX_MAP.inv[1], max_seq_length=max_seq_len, user_prompt="") for subj, msg in zip(subjects, messages)]

    # The first half are ham, the second half are spam
    prompts = prompts_ham + prompts_spam
    with torch.no_grad():
        seq_log_prob = get_seq_log_prob(prompts, tokenizer, model, device)

        '''
        Rearrange to (batch_size, 2), in this way, the second dimension 0 is ham, 1 is spam.
        '''
        seq_log_prob = rearrange(seq_log_prob, '(c b) -> b c', c=2)
        
        '''
        Apply softmax over ham/spam dimension to get probabilities.
        The shape of probs will be (2, batch_size), where probs[0, :] is ham probability and probs[1, :] is spam probability.
        probs[:, i] gives the category distribution used to classify spam and ham for the i-th email in the batch.
        '''
        probs = F.softmax(seq_log_prob, dim=-1)

        labels_pred = torch.argmax(probs, dim=-1)
        
        if  -1 in labels:
            is_correct = None
        else:
            is_correct = labels_pred.cpu() == labels

        return is_correct, (probs.detach().cpu(), labels_pred.detach().cpu())
    
    
def train_one_iter(model, tokenizer:Tokenizer, batch:torch.tensor, optimizer: Optimizer = None, max_seq_length=256,
                    is_training = True):
    '''
        Return 
        loss_val, is_correct, probs, labels_pred
        
        Where loss_val is the softmax CE loss
        is_correct[i] = labels_pred[i] == ground_truth[i]
        probs = probability vectors for predictions
    '''
    
    if is_training:
        model.train()
    else:
        model.eval()

    _, subjects, messages, label_indexs = batch
    
    if -1 in label_indexs:
        loss_val = None
    else:

        with torch.no_grad():
            spam_prompts = [get_prompt(subject=subj, message=msg, label=ENRON_LABEL_INDEX_MAP.inv[0], max_seq_length=max_seq_length) 
                                for subj, msg in zip(subjects, messages)]
            ham_prompts = [get_prompt(subject=subj, message=msg, label=ENRON_LABEL_INDEX_MAP.inv[1], max_seq_length=max_seq_length) 
                            for subj, msg in zip(subjects, messages)]

        # shape: 1xd for both spam and ham
        spam_seq_log_prob = get_seq_log_prob(spam_prompts, tokenizer, model, device=device).cpu()
        ham_seq_log_prob = get_seq_log_prob(ham_prompts, tokenizer, model, device=device).cpu()
    
        # Now, get the posterior probabilities for predicting spam and ham
        softmax_logits = torch.stack((spam_seq_log_prob, ham_seq_log_prob), dim=1)

        # pdb.set_trace()
        if -1 in label_indexs:
            loss_val = None
            
        ce_loss = torch.nn.CrossEntropyLoss()
        loss_val = ce_loss(softmax_logits, label_indexs)

        if is_training: 
            optimizer.zero_grad()
            loss_val.backward()
            optimizer.step()
            
        # Get prediction labels
        labels_pred = torch.argmax(softmax_logits, dim=-1)
        
        if -1 in label_indexs:
            is_correct = None
        else:
            is_correct = labels_pred == label_indexs
        
    # is_correct, (probs, labels_pred) = bayes_inverse_llm_classifier(args, model, batch, tokenizer, device=device)
    return loss_val, is_correct, (softmax_logits.detach().cpu(), labels_pred.detach().cpu())
    


In [9]:
def save_probs(args, model, tokenizer, dataloader, device, name = "test"):
    save_path = os.path.join(os.getcwd(), f"{args.prob_output_folder}/{name}_dataset_probs.csv")
    
    if os.path.exists(save_path):
        os.remove(save_path)
        
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="saving probabilities"):
            
            _, (probs, _) = bayes_inverse_llm_classifier(args, model, batch, tokenizer, device = device)
            data_index, _, _, _ = batch
            indices = torch.as_tensor(data_index).view(-1).tolist()
            
            rows = zip(indices, probs[:, 0].tolist(), probs[:, 1].tolist())
            
            file_exists = os.path.exists(save_path)
            with open(save_path, "a", newline="") as handle:
                if not file_exists:
                    handle.write("data_index,prob_ham,prob_spam\n")
                handle.writelines(f"{idx},{ham},{spam}\n" for idx, ham, spam in rows)


In [ ]:
class SpamClassifier:
    
    def __init__(self, n_models, config: Config, 
                    train_val_dataset: CPEN455_2025_W1_Dataset, 
                    checkpoint: str, model_cache_dir: str,
                    device=device):
    
        self.config = config    
        self.n_models = n_models
        self.models = [LlamaModel(config) for _ in range(n_models)]
        
        for model in self.models:
            load_model_weights(model, checkpoint, model_cache_dir)            
        
        training_dataset, val_dataset = prepare_subset(train_val_dataset, int(0.8 * len(train_val_dataset)), ratio_spam=0.5, return_remaining=True)
        
        self.training_dataset = training_dataset
        self.val_dataset = val_dataset
        self.device = device
        
        self.optimizers = [torch.optim.AdamW(model.parameters(), lr=1e-5) for model in self.models]
  
    
    def predict(self, tokenizer, batch, max_seq_length=256, return_loss=False):
        model_predictions = []

        # Do a prediction with each model
        with torch.no_grad():
            
            avg_loss = 0.0        
    
            for model in self.models:
                # model.to(self.device)
                val_loss, _, (_, label_pred) =  train_one_iter(model, tokenizer, 
                                            batch, is_training=False, 
                                            max_seq_length= max_seq_length)
                if return_loss:
                    avg_loss += val_loss
                    

                # Deallocate model
                # model.to("cpu")    
                # torch.cuda.empty_cache()  
                model_predictions.append(label_pred.to("cpu").unsqueeze(-1))
            
            avg_loss = avg_loss/self.n_models
            
            # Concatenate a ll of the model predictions, then vote on 
            # most frequent output
            model_predictions = torch.cat(model_predictions, dim=-1).mode(dim=-1).values
        
            if return_loss:
                return model_predictions, avg_loss
            else:
                return model_predictions
                

    
    def _evaluate_validation(self, val_dataloader: DataLoader, tokenizer,
                                max_seq_length=256) -> tuple[float, float]:
        '''
            Returns (avg_loss, avg_accuracy)
        '''

        val_loss_logger = avg_logger()
        val_accuracy_logger = avg_acc_logger()
        
        for val_batch in val_dataloader:
            _, _, _, label_index = val_batch 
            model_predictions, avg_loss = self.predict(
                                            tokenizer, val_batch, 
                                            max_seq_length, True)
            
            val_loss_logger.update(avg_loss)
            val_accuracy_logger.update(model_predictions.squeeze()  == label_index.squeeze() )
        
        return val_loss_logger.compute_average(), val_accuracy_logger.compute_accuracy()
        
    
    def train(self, tokenizer, num_iterations, batch_size, max_seq_length = 256):
        """Train all models with bagging (bootstrap sampling)"""
        from torch.utils.data import RandomSampler
        
        
        sampler = RandomSampler(
                            self.training_dataset, 
                            replacement=True, 
                            num_samples=len(self.training_dataset)
                        )
                
        train_loader = DataLoader(self.training_dataset, batch_size=batch_size, sampler=sampler)
        val_loader = DataLoader(self.val_dataset, batch_size=batch_size)
                    
        for iteration in tqdm(range(num_iterations), desc="Training"):
            if (iteration + 1) % (num_iterations // 10) == 0:
            
                # Evaluate on validation set
                with torch.no_grad():
                    avg_loss, avg_acc = self._evaluate_validation(val_loader, 
                                                                tokenizer, max_seq_length)
                    
                    wandb.log({
                        'val_avg_loss': avg_loss,
                        'val_avg_acc': avg_acc,
                        'training_iteration': iteration
                    })
            
            
            train_loss_logger = avg_logger()
            train_acc_logger = avg_acc_logger()
            train_iter = iter(train_loader)
            
            # Train each model independently, with bootstrap sampling    
            for model_idx, (model, optimizer) in enumerate(zip(self.models, self.optimizers)):
                # model.to(self.device)
                train_batch = next(train_iter)
                                
                model_loss_val, is_correct, _ = train_one_iter(
                    model, tokenizer, train_batch, optimizer, 
                )
                
                # model.to("cpu")
                # torch.cuda.empty_cache()  
                train_loss_logger.update(model_loss_val)
                train_acc_logger.update(is_correct)
    
            wandb.log({
                'average_loss': train_loss_logger.compute_average(),
                'avg_accuracy': train_acc_logger.compute_accuracy(), 
                'train_iteration': iteration
            })
    


In [11]:
run = wandb.init(
    project=os.getenv("PROJECT_NAME"),
    name=f"bayes-inverse-bagging_msl{max_seq_len}_ni{num_iterations}_bs{batch_size}_nmodel{num_models}",
)

checkpoint = os.getenv("MODEL_CHECKPOINT")
model_cache_dir = os.getenv("MODEL_CACHE_DIR")

tokenizer = Tokenizer.from_pretrained(checkpoint, cache_dir=model_cache_dir)
base_path = _resolve_snapshot_path(checkpoint, cache_dir=model_cache_dir)
config = Config._find_config_files(base_path)
train_val_dataset = CPEN455_2025_W1_Dataset(csv_path=dataset_path)
model = SpamClassifier(num_models, config, train_val_dataset, checkpoint, model_cache_dir)

model.train(tokenizer, num_iterations, batch_size, max_seq_len)




wandb: Currently logged in as: kenrickmh (kenrickmh-university-of-british-columbia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Found 1 snapshots in cache
Loading model weights from: ./cache/huggingface/transformers/models--HuggingFaceTB--SmolLM2-135M-Instruct/snapshots/12fd25f77366fa6b3b4b768ec3050bf629380bac/model.safetensors
Creating lm_head.weight from embed_tokens.weight
Missing keys: []
Unexpected keys: []
Found 1 snapshots in cache
Loading model weights from: ./cache/huggingface/transformers/models--HuggingFaceTB--SmolLM2-135M-Instruct/snapshots/12fd25f77366fa6b3b4b768ec3050bf629380bac/model.safetensors
Creating lm_head.weight from embed_tokens.weight
Missing keys: []
Unexpected keys: []
Found 1 snapshots in cache
Loading model weights from: ./cache/huggingface/transformers/models--HuggingFaceTB--SmolLM2-135M-Instruct/snapshots/12fd25f77366fa6b3b4b768ec3050bf629380bac/model.safetensors
Creating lm_head.weight from embed_tokens.weight
Missing keys: []
Unexpected keys: []


Training:   0%|          | 0/100 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 5.64 GiB of which 10.38 MiB is free. Including non-PyTorch memory, this process has 5.60 GiB memory in use. Of the allocated memory 5.44 GiB is allocated by PyTorch, and 56.74 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:

import os
import pdb
import wandb
from dotenv import load_dotenv
from einops import rearrange
from tqdm import tqdm
import argparse

import torch
from torch.utils.data import DataLoader
from torch.nn import functional as F

from autograder.dataset import CPEN455_2025_W1_Dataset, ENRON_LABEL_INDEX_MAP, prepare_subset
from model import LlamaModel
from utils.weight_utils import load_model_weights
from model.config import Config
from model.tokenizer import Tokenizer
from utils.download import _resolve_snapshot_path
from utils.device import set_device

from utils.prompt_template import get_prompt
from utils.logger import avg_logger, avg_acc_logger
    
load_dotenv()

checkpoint = os.getenv("MODEL_CHECKPOINT")
model_cache_dir = os.getenv("MODEL_CACHE_DIR")

run = None
    
# Set device to GPU if available, to MPS if on Mac with M-series chip, else CPU
device = set_device()
# device = "cpu"

# Load tokenizer and config
tokenizer = Tokenizer.from_pretrained(checkpoint, cache_dir=model_cache_dir)

base_path = _resolve_snapshot_path(checkpoint, cache_dir=model_cache_dir)
config = Config._find_config_files(base_path)

# Load model
model = LlamaModel(config)

load_model_weights(model, checkpoint, cache_dir=model_cache_dir, device=device)
model = model.to(device)

# Set up datasets and dataloaders
train_n_val_dataset = CPEN455_2025_W1_Dataset(csv_path=args.dataset_path)
training_dataset, val_dataset = prepare_subset(train_n_val_dataset, int(0.8 * len(train_n_val_dataset)), ratio_spam=0.5, return_remaining=True)
test_dataset = CPEN455_2025_W1_Dataset(csv_path=args.test_dataset_path)

training_dataloader = DataLoader(
    training_dataset, 
    batch_size=args.batch_size, 
    shuffle=True
    )

val_dataloader = DataLoader(
    val_dataset, 
    batch_size=args.batch_size, 
    shuffle=False
    )

test_dataloader = DataLoader(
    test_dataset, 
    batch_size=args.batch_size, 
    shuffle=False
    )

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

if os.path.exists(args.prob_output_folder) == False:
    os.makedirs(args.prob_output_folder)

for iteration in tqdm(range(args.num_iterations), desc="Training"):
    # Evaluate Validation Loss per 10 steaps
    if (iteration + 1) % 10 == 0:
        val_acc_logger = avg_acc_logger()
        val_loss_logger = avg_logger()
        
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Evaluating on validation set during training"):
                
                # bpd, is_correct, (probs, labels_pred) = train_or_test(
                #     args = args, 
                #     model = model, 
                #     tokenizer = tokenizer, 
                #     batch = batch, 
                #     is_training=False)
                
                loss_val, is_correct, (probs, labels_pred) = cross_entropy_test(
                    args = args, 
                    model = model, 
                    tokenizer = tokenizer, 
                    batch = batch, 
                    is_training=False)
                
                val_acc_logger.update(is_correct)
                val_loss_logger.update(loss_val.item())

                wandb.log({
                    "val_avg_bpd": val_loss_logger.compute_average(),
                    "val_avg_accuracy": val_acc_logger.compute_accuracy(),
                    "training_iteration": iteration,
                    })
                
    if not is_required_training(args.method):
        break
                
    batch = next(iter(training_dataloader))
    
    # Train on this batch
    loss_val, is_correct, _ = cross_entropy_test(
        args = args, 
        model = model, 
        tokenizer = tokenizer, 
        optimizer= optimizer,
        batch = batch, 
        is_training=True)
    
    wandb.log({
        "training_batch_bpd": loss_val.item(),
        "training_batch_acc": is_correct.float().mean().item(),
        "training_iteration": iteration,
        })

# After training, save probabilities on test set
train_n_val_dataloader = DataLoader(
    train_n_val_dataset, 
    batch_size=args.batch_size, 
    shuffle=False
    )
save_probs(args, model, tokenizer, train_n_val_dataloader, device=device, name = "train_n_val")
save_probs(args, model, tokenizer, test_dataloader, device=device, name = "test")

Using device: cuda
Found 1 snapshots in cache
Loading model weights from: ./cache/huggingface/transformers/models--HuggingFaceTB--SmolLM2-135M-Instruct/snapshots/12fd25f77366fa6b3b4b768ec3050bf629380bac/model.safetensors
Creating lm_head.weight from embed_tokens.weight
Missing keys: []
Unexpected keys: []


NameError: name 'args' is not defined